# GAIA embeddings by modality (Plotly)

Interactive visualization of graph-level embeddings for `metric`, `log`, and `trace` with color by stage (`before`/`after`).

## Stage definition for GAIA

Stage is derived directly from `label.csv` timestamps:
- `before`: `datetime < st_time`
- `after`: `datetime >= st_time`

So coloring is based on incident time, not train/test split.

In [1]:
!pip install torch==2.1.1
!pip install dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html
!pip install scikit-learn
!pip install fasttext==0.9.2

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 9.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 76.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 88.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 117.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 72.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 16.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 56.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import torch

import plotly.express as px
import plotly.io as pio

from config.exp_config import Config
from core.model.MainModel import MainModel
from process.EventProcess import EventProcess
from helper.logger import get_logger

pio.renderers.default = 'notebook_connected'
torch.set_grad_enabled(False)

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


In [3]:
# ===== User settings =====
DATASET = 'gaia'
LOG_DIR = f'./logs/{DATASET}'
CHECKPOINT_PATH = os.path.join(LOG_DIR, 'TVDiag.pt')

TIME_COL = 'datetime'
INCIDENT_START_COL = 'st_time'
REDUCER = 'umap'  # umap | tsne
RANDOM_STATE = 42

assert DATASET == 'gaia', 'This notebook cell is configured for GAIA timestamp stage splitting.'
assert os.path.exists(CHECKPOINT_PATH), f'Checkpoint not found: {CHECKPOINT_PATH}'


In [4]:
# Load config, model and data
config = Config(DATASET)
config.reconstruct = False

logger = get_logger(LOG_DIR, f'{DATASET}_embedding_viz')
processor = EventProcess(config, logger)
train_data, _, test_data = processor.process(reconstruct=config.reconstruct)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MainModel(config).to(device)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()

print('Dataset:', DATASET)
print('Device:', device)
print('Train samples:', len(train_data), 'Test samples:', len(test_data))

/workdir/TVDiag/core/multimodal_dataset.py:14: UserWarning:

Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:261.)



Dataset: gaia
Device: cuda
Train samples: 160 Test samples: 939


In [5]:
# Build metadata aligned with train_data / test_data order
labels = pd.read_csv(f'./data/{DATASET}/label.csv').copy()
labels['index'] = labels['index'].astype(str)

required_cols = [TIME_COL, INCIDENT_START_COL]
missing = [c for c in required_cols if c not in labels.columns]
if missing:
    raise ValueError(f'Missing required columns in label.csv: {missing}')

labels[TIME_COL] = pd.to_datetime(labels[TIME_COL], errors='coerce')
labels[INCIDENT_START_COL] = pd.to_datetime(labels[INCIDENT_START_COL], errors='coerce')
if labels[TIME_COL].isna().any() or labels[INCIDENT_START_COL].isna().any():
    bad = labels[labels[TIME_COL].isna() | labels[INCIDENT_START_COL].isna()][['index', TIME_COL, INCIDENT_START_COL]].head(5)
    raise ValueError(f'Failed to parse datetime columns. Examples:\n{bad}')

labels['stage'] = np.where(labels[TIME_COL] < labels[INCIDENT_START_COL], 'before', 'after')

meta_train = labels[labels['data_type'] == 'train'].reset_index(drop=True).copy()
meta_test = labels[labels['data_type'] == 'test'].reset_index(drop=True).copy()

assert len(meta_train) == len(train_data), (len(meta_train), len(train_data))
assert len(meta_test) == len(test_data), (len(meta_test), len(test_data))

meta_all = pd.concat([meta_train, meta_test], axis=0).reset_index(drop=True)
print('Stage distribution:', meta_all['stage'].value_counts().to_dict())
delta_sec = (meta_all[TIME_COL] - meta_all[INCIDENT_START_COL]).dt.total_seconds()
print('delta_sec stats (datetime - st_time):')
print(delta_sec.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

Stage distribution: {'before': 1095, 'after': 4}
delta_sec stats (datetime - st_time):
count     1099.000000
mean    -42731.627966
std      24957.641462
min     -86329.175000
10%     -76850.064102
25%     -64823.539444
50%     -42820.195000
75%     -21534.919776
90%      -8609.195200
max        570.541477
dtype: float64


In [6]:
# Extract graph-level embeddings per modality
def collect_embeddings(dataset, meta_df, split_name):
    rows = []
    for i, (graph, _) in enumerate(dataset):
        graph = graph.to(device)
        meta = meta_df.iloc[i]
        for modality in config.modalities:
            x = graph.ndata[modality]
            f, _ = model.encoders[modality](graph, x)
            rows.append({
                'split': split_name,
                'index': str(meta['index']),
                'anomaly_type': str(meta['anomaly_type']),
                'stage': str(meta['stage']),
                'modality': modality,
                'embedding': f.squeeze(0).detach().cpu().numpy(),
            })
    return rows

rows = []
rows += collect_embeddings(train_data, meta_train, 'train')
rows += collect_embeddings(test_data, meta_test, 'test')
emb_df = pd.DataFrame(rows)

print('Embeddings rows:', len(emb_df))
print('Modalities:', emb_df['modality'].unique().tolist())
print('Stage distribution:', emb_df['stage'].value_counts().to_dict())

Embeddings rows: 3297
Modalities: ['metric', 'trace', 'log']
Stage distribution: {'before': 3285, 'after': 12}


In [7]:
# 2D reducer
def reduce_2d(X, reducer='umap', random_state=42):
    if reducer == 'umap':
        try:
            import umap
            model_2d = umap.UMAP(n_components=2, random_state=random_state)
            return model_2d.fit_transform(X)
        except Exception as e:
            print('UMAP unavailable, fallback to t-SNE:', e)

    from sklearn.manifold import TSNE
    perp = min(30, max(5, X.shape[0] // 20))
    model_2d = TSNE(n_components=2, random_state=random_state, init='pca', perplexity=perp)
    return model_2d.fit_transform(X)

In [8]:
# Plotly interactive plots: metric / log / trace
modalities = ['metric', 'log', 'trace']
color_map = {'before': '#1f77b4', 'after': '#d62728'}

for modality in modalities:
    sub = emb_df[emb_df['modality'] == modality].reset_index(drop=True)
    X = np.stack(sub['embedding'].values)
    Z = reduce_2d(X, reducer=REDUCER, random_state=RANDOM_STATE)

    plot_df = pd.DataFrame({
        'x': Z[:, 0],
        'y': Z[:, 1],
        'stage': sub['stage'].values,
        'anomaly_type': sub['anomaly_type'].values,
        'index': sub['index'].values,
        'split': sub['split'].values,
    })

    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='stage',
        color_discrete_map=color_map,
        hover_data=['index', 'split', 'anomaly_type', 'stage'],
        title=f'GAIA {modality} embeddings (before/after by incident time, reducer={REDUCER})',
        opacity=0.8,
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(height=520, width=880)
    fig.show()


UMAP unavailable, fallback to t-SNE: No module named 'umap'


UMAP unavailable, fallback to t-SNE: No module named 'umap'


UMAP unavailable, fallback to t-SNE: No module named 'umap'
